# 01 - EDA fraude bancaire

Objectif: analyser les transactions, les montants, le desequilibre de classe et les comportements suspects avant modelisation.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from ml_project.data.loaders import read_fraud_transactions

df = read_fraud_transactions()
df.head()

In [ ]:
# Controle qualite et desequilibre de la cible
display(df.info())
display(df.isna().sum())
fraud_rate = df['isFraud'].mean()
print(f'Transactions: {len(df):,}')
print(f'Fraudes: {df.isFraud.sum():,}')
print(f'Taux de fraude: {fraud_rate:.4%}')

In [ ]:
# Risque par type de transaction
type_summary = (
    df.groupby('type')
    .agg(transactions=('isFraud', 'size'), fraudes=('isFraud', 'sum'), montant_median=('amount', 'median'))
    .assign(taux_fraude=lambda x: x['fraudes'] / x['transactions'])
    .sort_values('taux_fraude', ascending=False)
)
display(type_summary)
type_summary['taux_fraude'].plot(kind='bar', title='Taux de fraude par type')

In [ ]:
# Analyse des montants frauduleux
fraud_only = df[df['isFraud'] == 1]
display(fraud_only['amount'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
fraud_only.sort_values('amount', ascending=False).head(20)

Conclusion metier: le risque est rare mais concentre. Les controles doivent se concentrer sur les types frauduleux et sur les montants les plus exposes, sans bloquer les transactions normales.